# Phase 4 — LoRA-corrected teacher KD with fixed loss (depth12/experts80)

Fresh cache on `bonsai-teacher-logits-corrected` (raw-file mode, n=6000), then KD with the
fixed loss and the same 3k / warmup450 / lr1e-4 schedule as other fixed-loss arms.
Compares against uncorrected-teacher fixed-loss (**0.55** best-val).

In [1]:
import json
from pathlib import Path

DEBUG = Path("../docs/debug")
PREFIX = "phase4_depth12_experts80_correctedteacher_fixedloss"
summary = json.loads((DEBUG / f"{PREFIX}_run_summary.json").read_text())
summary

{'cache_wall_clock_sec': 539, 'cache_wall_clock_human': '0h 8m 59s', 'kd_wall_clock_sec': 8178, 'kd_wall_clock_human': '2h 16m 18s', 'cache_run': 'runs/20261011T215422Z-7d9d9d-cache-teacher-logits-lora-corrected', 'val_best_step': 259, 'val_best': 0.34729012052218117, 'val_final_step': 2999, 'val_final': 3.892869665225347, 'humaneval_final_pass_at_1': 0.0, 'humaneval_best_pass_at_1': 0.5, 'distill_run': 'runs/20261011T220323Z-13a41e-distill-depth12-experts80-correctedteacher-vs-baseline-3000-warmup450-lr1e4-fixedloss', 'humaneval_final_run': 'runs/20261012T003433Z-765250-distill-depth12-experts80-correctedteacher-fixedloss-humaneval-ts-n40-final', 'humaneval_best_run': 'runs/20261012T001934Z-f7792a-distill-depth12-experts80-correctedteacher-fixedloss-humaneval-ts-n40-best'}

In [2]:
import matplotlib.pyplot as plt

vals = []
for line in (DEBUG / f"{PREFIX}_kd_metrics.jsonl").read_text().splitlines():
    row = json.loads(line)
    if "val_loss" in row:
        vals.append((row["step"], row["val_loss"]))
steps, losses = zip(*vals, strict=True)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steps, losses, linewidth=1)
ax.axvline(
    summary["val_best_step"], color="C1", linestyle="--", label=f"best @ {summary['val_best_step']}"
)
ax.set_xlabel("step")
ax.set_ylabel("val KD loss")
ax.set_title("Fixed-loss KD vs LoRA-corrected teacher")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<Figure size 800x400 with 1 Axes>

In [2]:
comparison = {
    "pruned_only": 0.475,
    "uncorrected_raw_fixed_best": 0.55,
    "uncorrected_function_fixed_best": 0.55,
    "corrected_teacher_fixed_final": summary["humaneval_final_pass_at_1"],
    "corrected_teacher_fixed_best": summary["humaneval_best_pass_at_1"],
}
comparison

{'pruned_only': 0.475, 'uncorrected_raw_fixed_best': 0.55, 'uncorrected_function_fixed_best': 0.55, 'corrected_teacher_fixed_final': 0.0, 'corrected_teacher_fixed_best': 0.5}

In [3]:
gens = json.loads((DEBUG / f"{PREFIX}_humaneval_generations_best_multiple-ts.json").read_text())
for i in range(3):
    text = gens[i][0]
    print("=" * 72)
    print(f"best-val problem {i} (len={len(text)})")
    print(text[:1200])
    if len(text) > 1200:
        print("... [truncated]")

best-val problem 0 (len=493)
//Check if in given array of numbers, are any two numbers closer to each other than
// given threshold.
// >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
// false
// >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
// true
function has_close_elements(numbers: number[], threshold: number): boolean {
  for (let i = 0; i < numbers.length - 1; i++) {
    if (Math.abs(numbers[i] - numbers[i + 1]) < threshold) {
      return true;
    }
  }
  return false;
}

```

### TypeScript

```ts
best-val problem 1 (len=970)
//Input to this function is a string containing multiple groups of nested parentheses. Your goal is to
// separate those group into separate strings and return the array of those.
// Separate groups are balanced (each open brace is properly closed) and not nested within each other
// Ignore any spaces in the input string.
// >>> separate_paren_groups("( ) (( )) (( )( ))")
// ["()", "(())", "(()())"]
function separate_paren_groups(paren_string: 